# M5 Tabular Shift Study — Kaggle Runner

**Inputs attached:**
- Project: `/kaggle/input/datasets/subicharan/python-files/walmart-tabular-shift/`
- M5 data: `/kaggle/input/competitions/m5-forecasting-accuracy/`

| Cell | What it does | Time |
|------|-------------|------|
| 1 | Install deps + add `shift_study` to path | ~2 min |
| 2 | Copy configs, verify data & GPU | <1 min |
| 3 | Build `features.parquet` | ~15 min |
| 4 | Smoke test (LightGBM E2, 500 series) | ~2 min |
| 5 | Clean smoke artefacts | instant |
| 6 | **Session 1:** XGBoost / LightGBM / TabPFN / TabM | ~4–6 h |
| 7 | **Session 2:** TabR only (*Save & Run All*) | ~8–10 h |
| 8 | Crossover curve + report | ~1 h |
| 9 | Display results inline | instant |
| 10 | List all output files | instant |

> Re-run **Cells 1–3** at the start of every new Kaggle session. `--skip-existing` means a crashed session loses nothing already done.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os

PROJ = "/kaggle/input/datasets/subicharan/python-files/walmart-tabular-shift"

# 1) Install third-party requirements (read-only path is fine for -r)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", f"{PROJ}/requirements.txt",
])

# 2) Make shift_study importable — no build needed, just add src/ to path
src = f"{PROJ}/src"
if src not in sys.path:
    sys.path.insert(0, src)

# 3) Also expose for every subprocess spawned later
os.environ["PYTHONPATH"] = src + ":" + os.environ.get("PYTHONPATH", "")

import shift_study  # smoke-import to confirm it works
print(f"\n✅ shift_study importable from {src}")

## Cell 2 — Setup & Verify

In [4]:
import os, shutil, torch
from pathlib import Path

PROJ    = "/kaggle/input/datasets/subicharan/python-files/walmart-tabular-shift"
RAW_DIR = "/kaggle/input/competitions/m5-forecasting-accuracy"

# copy configs → /kaggle/working/configs/ so scripts find 'configs/*.yaml'
dst = Path("/kaggle/working/configs")
dst.mkdir(exist_ok=True)
for f in (Path(PROJ) / "configs").glob("*.yaml"):
    shutil.copy(f, dst / f.name)
    print(f"  copied configs/{f.name}")

os.environ["SS_PROJ"]    = PROJ
os.environ["SS_RAW_DIR"] = RAW_DIR

print("\n=== Data files ===")
for fname in ["calendar.csv", "sell_prices.csv",
              "sales_train_validation.csv",
              "sales_train_evaluation.csv",
              "sample_submission.csv"]:
    p = Path(RAW_DIR) / fname
    status = f"✅ {p.stat().st_size/1e6:.0f} MB" if p.exists() else "❌ MISSING"
    print(f"  {fname:<45s} {status}")

print("\n=== GPU ===")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  {torch.cuda.get_device_name(0)}  {props.total_memory/1e9:.1f} GB")

print("\n✅ Setup complete.")

  copied configs/xgboost.yaml
  copied configs/tabpfn.yaml
  copied configs/tabm.yaml
  copied configs/lightgbm.yaml
  copied configs/tabr.yaml

=== Data files ===
  calendar.csv                                  ✅ 0 MB
  sell_prices.csv                               ✅ 203 MB
  sales_train_validation.csv                    ✅ 120 MB
  sales_train_evaluation.csv                    ✅ 122 MB
  sample_submission.csv                         ✅ 5 MB

=== GPU ===
  CUDA: True
  Tesla T4  15.6 GB

✅ Setup complete.


## Cell 3 — Build Feature Table (~15 min)

In [5]:
import subprocess, os, sys
PROJ    = os.environ["SS_PROJ"]
RAW_DIR = os.environ["SS_RAW_DIR"]

env = {**os.environ, "PYTHONPATH": f"{PROJ}/src:" + os.environ.get("PYTHONPATH", "")}

subprocess.check_call([
    sys.executable, f"{PROJ}/scripts/build_features.py",   # sys.executable, not "python"
    "--raw-dir", RAW_DIR,
    "--out",     "/kaggle/working/features.parquet",
], cwd="/kaggle/working", env=env)

Building base table...
  rows=46,027,957  series=30,490  days=1..1913
Adding features...
  rows after NaN-lag drop=44,960,807  features=23
Wrote /kaggle/working/features.parquet (362 MB) in 304s


0

## Cell 4 — Smoke Test

In [6]:
import subprocess, os, sys
PROJ = os.environ["SS_PROJ"]
env  = {**os.environ, "PYTHONPATH": f"{PROJ}/src:" + os.environ.get("PYTHONPATH", "")}

subprocess.check_call([
    sys.executable, f"{PROJ}/scripts/run_experiment.py",
    "--model", "lightgbm", "--experiment", "e2",
    "--features", "/kaggle/working/features.parquet",
    "--out",      "/kaggle/working/results",
    "--sample",   "500",
], cwd="/kaggle/working", env=env)

Loaded 742,839 rows, 500 series
train=487,245 val=60,011 test=195,583
Training until validation scores don't improve for 75 rounds
[100]	valid_0's l1: 0.741407
[200]	valid_0's l1: 0.740066
Early stopping, best iteration is:
[191]	valid_0's l1: 0.740007
  seed=42  WAPE=0.7091
WAPE = 0.7091 ± 0.0000  -> /kaggle/working/results/lightgbm_e2.json


0

## Cell 5 — Clean Smoke Artefacts

In [7]:
import glob, os

for f in glob.glob("/kaggle/working/results/lightgbm_e2*"):
    os.remove(f)
    print(f"removed {f}")

print("✅ Ready for full study.")

removed /kaggle/working/results/lightgbm_e2.json
removed /kaggle/working/results/lightgbm_e2_items.parquet
✅ Ready for full study.


## Cell 6 — Full Study: Session 1
XGBoost / LightGBM / TabPFN / TabM × E1/E2/E3. Resumable — rerun after any session death.

> ⚠️ TabR excluded here — run it in Cell 7 in a dedicated session.

In [8]:
import subprocess, itertools, time, os, sys
PROJ = os.environ["SS_PROJ"]
env  = {**os.environ, "PYTHONPATH": f"{PROJ}/src:" + os.environ.get("PYTHONPATH", "")}

t0 = time.time()
for model, exp in itertools.product(
    ["xgboost", "lightgbm", "tabpfn", "tabm"], ["e1", "e2", "e3"]
):
    print(f"\n{'='*55}\n  ▶  {model} × {exp}\n{'='*55}")
    subprocess.run([
        sys.executable, f"{PROJ}/scripts/run_experiment.py",
        "--model", model, "--experiment", exp,
        "--features", "/kaggle/working/features.parquet",
        "--out", "/kaggle/working/results",
        "--skip-existing",
    ], cwd="/kaggle/working", env=env)

print(f"\n✅ Session 1 done in {(time.time()-t0)/3600:.1f} h")


  ▶  xgboost × e1
Loaded 44,960,807 rows, 30,490 series
train=31,472,566 val=4,496,080 test=8,992,161
[0]	validation_0-mae:1.39573


KeyboardInterrupt: 

## Cell 7 — TabR (Dedicated Session 2, ~8–10 h)
Start a fresh session, run Cells 1–3, then run **only this cell** with *Persistence → Save & Run All*.

In [ ]:
import subprocess, time, os
PROJ = os.environ["SS_PROJ"]

t0 = time.time()
for exp in ["e1", "e2", "e3"]:
    print(f"\n{'='*55}\n  ▶  tabr × {exp}\n{'='*55}")
    subprocess.run([
        "python", f"{PROJ}/scripts/run_experiment.py",
        "--model",      "tabr",
        "--experiment", exp,
        "--features",   "/kaggle/working/features.parquet",
        "--out",        "/kaggle/working/results",
        "--skip-existing",
    ], cwd="/kaggle/working")

print(f"\n✅ TabR done in {(time.time()-t0)/3600:.1f} h")

## Cell 8 — Crossover Curve + Report

In [ ]:
import subprocess, os
PROJ = os.environ["SS_PROJ"]

subprocess.check_call([
    "python", f"{PROJ}/scripts/run_crossover.py",
    "--features", "/kaggle/working/features.parquet",
    "--out",      "/kaggle/working/results",
], cwd="/kaggle/working")

subprocess.check_call([
    "python", f"{PROJ}/scripts/make_report.py",
    "--results", "/kaggle/working/results",
], cwd="/kaggle/working")

## Cell 9 — Display Results

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
from pathlib import Path

REPORT = Path("/kaggle/working/results/report")

mt = REPORT / "master_table.csv"
if mt.exists():
    display(Markdown("### Master Table — WAPE & Δ"))
    display(pd.read_csv(mt).style.format(precision=4))
else:
    print("⚠️  Run Cell 8 first.")

for fname, title in [
    ("fig1_shift_sensitivity.png", "Figure 1: Shift Degradation"),
    ("fig2_crossover.png",         "Figure 2: Cold-Start Crossover"),
]:
    p = REPORT / fname
    if p.exists():
        display(Markdown(f"### {title}"))
        display(Image(str(p), width=720))
    else:
        print(f"⚠️  {fname} not found.")

wt = REPORT / "wilcoxon.csv"
if wt.exists():
    display(Markdown("### Wilcoxon Signed-Rank Tests"))
    display(pd.read_csv(wt).style.format(precision=4))

## Cell 10 — List Output Files

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/"):
    dirs[:] = [d for d in sorted(dirs) if d != "configs"]
    lvl    = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * lvl
    print(f"{indent}{os.path.basename(root) or 'working'}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        unit = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.0f} KB"
        print(f"{indent}  {f}  ({unit})")